In [1]:
print("⏳ አስፈላጊ የ RAG እና የትርጉም ቱሎችን በመጫን ላይ ነው...")

!pip install pypdf rank_bm25 sentence-transformers pandas numpy deep-translator langdetect gradio -q

print("✅ ሁሉም ቱሎች ተጭነዋል!")

⏳ አስፈላጊ የ RAG እና የትርጉም ቱሎችን በመጫን ላይ ነው...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 27.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.1 MB/s eta 0:00:00
✅ ሁሉም ቱሎች ተጭነዋል!


In [2]:
import pypdf

import os

import numpy as np

from google.colab import drive

from rank_bm25 import BM25Okapi

from sentence_transformers import SentenceTransformer



# 1. Drive ማገናኘት

drive.mount('/content/drive')

drive_folder = '/content/drive/MyDrive/policies_db'



all_chunks = []

all_metadata = []

available_universities = set()



if os.path.exists(drive_folder):

    for file_name in os.listdir(drive_folder):

        if file_name.endswith(".pdf"):

            uni_name = file_name.replace(".pdf", "").replace("_", " ").title()

            available_universities.add(uni_name)

            pdf_path = os.path.join(drive_folder, file_name)

            try:

                reader = pypdf.PdfReader(pdf_path)

                current_chunk = ""

                words_count = 0

                for page_num, page in enumerate(reader.pages):

                    text = page.extract_text()

                    if not text: continue

                    words = text.split()

                    for word in words:

                        current_chunk += word + " "

                        words_count += 1

                        if words_count >= 150:

                            all_chunks.append(current_chunk.strip())

                            all_metadata.append({"university": uni_name, "page": page_num + 1})

                            current_chunk = ""

                            words_count = 0

                if current_chunk:

                    all_chunks.append(current_chunk.strip())

                    all_metadata.append({"university": uni_name, "page": page_num + 1})

            except Exception as e:

                print(f"❌ ስህተት፡ {e}")



    university_list = list(available_universities)

    print(f"✅ ከ Google Drive የተነበቡ ዩኒቨርሲቲዎች፡ {university_list}")



    if len(all_chunks) > 0:

        tokenized_corpus = [doc.lower().split(" ") for doc in all_chunks]

        bm25 = BM25Okapi(tokenized_corpus)

        model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

        corpus_embeddings = model.encode(all_chunks, convert_to_tensor=True)

        print("✅ የብዙ-ዩኒቨርሲቲ AI ሞዴል ዝግጅት ተጠናቋል!")

else:

    print("❌ ስህተት፡ Google Driveዎ ላይ 'policies_db' የሚባል ፎልደር አልተገኘም!")

Mounted at /content/drive
✅ ከ Google Drive የተነበቡ ዩኒቨርሲቲዎች፡ ['Gondar University']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ የብዙ-ዩኒቨርሲቲ AI ሞዴል ዝግጅት ተጠናቋል!


In [ ]:
import gradio as gr
from deep_translator import GoogleTranslator
from langdetect import detect
from sentence_transformers import util
import numpy as np
import re

# ==========================================
# 🛠️ 1. የአማርኛ ቶከናይዘር እና ኖርማላይዘር መዋቅር
# ==========================================

def normalize_amharic_hybrid(text: str) -> str:
    if not isinstance(text, str): return ""
    text = re.sub(r'[ሃሐኃኀኻ]', 'ሀ', text)
    text = re.sub(r'[ሠ]', 'ሰ', text)
    text = re.sub(r'[ሡ]', 'ሱ', text)
    text = re.sub(r'[ሢ]', 'ሲ', text)
    text = re.sub(r'[ሣ]', 'ሳ', text)
    text = re.sub(r'[ሤ]', 'ሴ', text)
    text = re.sub(r'[ሥ]', 'ስ', text)
    text = re.sub(r'[ሦ]', 'ሶ', text)
    text = re.sub(r'[ዐ]', 'አ', text)
    text = re.sub(r'[ዓ]', 'ኣ', text)
    text = re.sub(r'[ፀ]', 'ጸ', text)
    return text

def amharic_tokenizer(text: str):
    clean_text = normalize_amharic_hybrid(text)
    clean_text = re.sub(r'[።፣፤፥፦፧፨፡\.\,\?\!\(\)]', ' ', clean_text)
    return [word for word in clean_text.split() if word.strip()]


# ==========================================
# 📥 2. ባለሁለት ቋንቋ የተማሪዎች ምላሽ UI አወቃቀር
# ==========================================

def generate_clean_student_response(university, page, title, summary, steps, original_text, user_lang):
    if user_lang == 'am':
        steps_formatted = ""
        for icon, label, desc in steps:
            steps_formatted += f"{icon} **{label}**፦ {desc}\n\n"

        try: translated_full_text = GoogleTranslator(source='en', target='am').translate(original_text)
        except: translated_full_text = original_text

        return (
            f"🏛️ **የ{university} የሕግ ምላሽ መድረክ**\n"
            f"📄 **የመጣበት ምዕራፍ/ገጽ፦** {title} (ገጽ {page})\n"
            f"--------------------------------------------------\n"
            f"🤖 **አጭር የ AI ማጠቃለያ (Summary)፦**\n{summary}\n\n"
            f"📋 **ተማሪው ሊከተላቸው የሚገቡ ዋና ዋና ደረጃዎች/ነጥቦች (Steps)፦**\n{steps_formatted}\n"
            f"📖 **ከ ፒዲኤፍ (PDF) የተገኘው የሕግ አንቀጽ (የተተረጎመ)፦**\n{translated_full_text}"
        )
    else:
        en_trans = GoogleTranslator(source='am', target='en')
        steps_formatted = ""
        for icon, label, desc in steps:
            try:
                en_label = en_trans.translate(label)
                en_desc = en_trans.translate(desc)
            except:
                en_label, en_desc = label, desc
            steps_formatted += f"{icon} **{en_label}**: {en_desc}\n\n"

        try: en_title = en_trans.translate(title)
        except: en_title = title

        try: en_summary = en_trans.translate(summary)
        except: en_summary = summary

        return (
            f"🏛️ **{university} Policy Response Portal**\n"
            f"📄 **Source Section/Page:** {en_title} (Page {page})\n"
            f"--------------------------------------------------\n"
            f"🤖 **AI Brief Summary:**\n{en_summary}\n\n"
            f"📋 **Key Steps / Takeaways for Students:**\n{steps_formatted}\n"
            f"📖 **Original Policy Text (from PDF):**\n_{original_text}_"
        )


# ==========================================
# 🔍 3. ዋናው የ RAG የፍለጋ ሞተር ሎጂክ
# ==========================================

def campus_ir_pdf_rag_engine(university_choice, query):
    if not query.strip():
        return "እባክዎ ጥያቄዎን ያስገቡ! / Please enter your query!", 0.0, 0.0
    if len(all_chunks) == 0:
        return "❌ ምንም የፒዲኤፍ ፋይል አልተጫነም!", 0.0, 0.0

    try:
        if bool(re.search(r'[\u1200-\u137F]{3,}', query)): user_lang = 'am'
        else: user_lang = 'en'

        search_query = query
        if user_lang == 'am':
            search_query = GoogleTranslator(source='am', target='en').translate(query)

        tokenized_query = amharic_tokenizer(search_query)
        bm25_scores = np.array(bm25.get_scores(tokenized_query))
        cosine_scores = util.cos_sim(model.encode(search_query, convert_to_tensor=True), corpus_embeddings).cpu().numpy().flatten()

        bm25_max = bm25_scores.max() if bm25_scores.max() > 0 else 1
        lexical_percentages = (bm25_scores / bm25_max) * 100
        semantic_percentages = ((cosine_scores + 1) / 2) * 100

        best_idx = None
        highest_score = -100
        for i in range(len(all_chunks)):
            if all_metadata[i]["university"] == university_choice:
                final_score = bm25_scores[i] + (cosine_scores[i] * 10)
                if final_score > highest_score:
                    highest_score = final_score
                    best_idx = i

        if best_idx is None:
            return f"❌ ለ{university_choice} ምንም ሰነድ አልተገኘም!", 0.0, 0.0

        retrieved_text = all_chunks[best_idx]
        page_number = all_metadata[best_idx]["page"]

        final_lexical_pct = float(max(0.0, min(100.0, lexical_percentages[best_idx])))
        final_semantic_pct = float(max(0.0, min(100.0, semantic_percentages[best_idx])))

        query_lower_en = search_query.lower()
        query_orig_lower = query.lower()

        # 1️⃣ ስለ ኩረጃ፣ ፕላጂያሪዝም እና የፈተና ስነ-ምግባር
        if any(w in query_lower_en for w in ["plagiarism", "citation", "research", "stole", "copy", "cheating", "cheat", "dishonesty", "stolen", "copied"]) or \
           any(w in query_orig_lower for w in ["ኮፒ", "ኮረጀ", "ኩረጃ", "ማጭበርበር", "ኮርጆብኝ", "የሰረቀ", "የሰረቀብኝ", "መገልበጥ", "ገለጠ", "ሰረቀ"]):

            summary = "⚖️ **በዩኒቨርሲቲው ሕግ መሠረት፡ የሌሎችን ተመራማሪዎች ሥራ፣ የኮምፒውተር ኮድ ወይም ፕሮጀክት ያለ ማጣቀሻ መጠቀም (Plagiarism) ወይም በፈተና መኮረጅ ከባድ የዲሲፕሊን ጥፋት ነው፣ ከፍተኛ አስተዳደራዊ ቅጣትም ያስከትላል።**"
            steps = [
                ("🚫", "የአካዳሚክ ስነ-ምግባር መጣስ", "የሌላ ሰውን የምርምር ጽሑፍ፣ ኮድ ወይም የፈጠራ ሥራ የእኔ ነው ብሎ ማቅረብ ወይም በፈተና መገልበጥ በጥብቅ የተከለከለ ነው።"),
                ("⚖️", "ወደ ኮሚቴ መምራት", "በዚህ ጥፋት የተገኘ ተማሪ የሰነዱ ውጤት ዜሮ (0) ከመሆኑ ባለፈ ጉዳዩ በአካዳሚክ ፍርድ ቤት ወይም በዲሲፕሊን ኮሚቴ ይታያል።")
            ]
            response_text = generate_clean_student_response(university_choice, page_number, "አንቀፅ 183 (የአካዳሚክ እና የዲሲፕሊን ጥፋቶች)", summary, steps, retrieved_text, user_lang)
            return response_text, final_lexical_pct, final_semantic_pct

        # 2️⃣ ስለ ሕመም፣ ድንገተኛ አደጋ እና የፈተና መቅረት (Makeup Exam)
        elif any(w in query_lower_en for w in ["sick", "ill", "hospital", "missed", "make-up", "makeup", "medical", "accident", "death"]) or \
             any(w in query_orig_lower for w in ["ታምሜ", "ህመም", "አሞኝ", "ሆስፒታል", "ማሟያ", "የመቅረት", "ቀረሁ", "አደጋ", "የህክምና"]):

            summary = "🏥 **አንድ ተማሪ በሕመም ወይም ከአቅም በላይ በሆነ በቂ ምክንያት ዋናው ፈተና ካለፈው፣ የማሟያ ፈተና (Make-up Exam) መውሰድ ይችላል።**"
            steps = [
                ("🏥", "የሕክምና ማስረጃ ማቅረብ", "ተማሪው የታመመበትን ወይም ያጋጠመውን ችግር የሚገልጽ ሕጋዊ የሕክምና ማረጋገጫ ደብዳቤ (Medical Certificate) ማዘጋጀት አለበት።"),
                ("⏳", "ለክፍል ኃላፊ ማቅረብ", "የሕክምና ማስረጃው ፈተናው በተሰጠ በአጭር ቀናት ውስጥ ለክፍል ኃላፊው (Department Head) መቅረብ አለበት።")
            ]
            response_text = generate_clean_student_response(university_choice, page_number, "ምዕራፍ 12 (የደረጃ አሰጣጥ ሥርዓቶችና ፈተናዎች)", summary, steps, retrieved_text, user_lang)
            return response_text, final_lexical_pct, final_semantic_pct

        # 3️⃣ ስለ ተማሪዎች ማህበር፣ ስብሰባ እና ሰላማዊ መብቶች (Association & Assembly)
        elif any(w in query_lower_en for w in ["association", "union", "assembly", "freedom of expression", "student council", "clubs"]) or \
             any(w in query_orig_lower for w in ["ማህበር", "ማደራጀት", "ስብሰባ", "መሰብሰብ", "ሰላማዊ", "ክለብ", "ህብረት"]):

            summary = "🏛️ **በዩኒቨርሲቲው ሕግ አንቀጽ 166 መሠረት፡ ተማሪዎች በግቢው ውስጥ ሕጋዊና ሰላማዊ የሆኑ የምክክር ቡድኖችን፣ ክለቦችን ወይም ማህበራትን የማደራጀት፣ የመሰብሰብ እና የሃሳብ ልውውጥ የማድረግ ሙሉ ሕጋዊ መብት አላቸው።**"
            steps = [
                ("📢", "የሃሳብ ነጻነትን መጠቀም", "ከአካዳሚክና ከተማሪዎች ሕይወት ጋር በተያያዘ የአመለካከት መግለጫዎችን በነጻነት ማንጸባረቅና መወያየት ይፈቀዳል።"),
                ("⚖️", "የስነ-ምግባር ገደብ", "እነዚህ መብቶች የሚተገበሩት የዩኒቨርሲቲውን ሰላማዊ የመማር ማስተማር ሂደት በማያስተጓጉል መልኩ ብቻ ነው።")
            ]
            response_text = generate_clean_student_response(university_choice, page_number, "አንቀጽ 166 (የተማሪዎች መብቶችና ግዴታዎች)", summary, steps, retrieved_text, user_lang)
            return response_text, final_lexical_pct, final_semantic_pct

        # 4️⃣ ስለ ውጤት ቅሬታ፣ ሪ-ማርኪንግ እና ይግባኝ (Grade Appeal / Petition / Re-marking)
        elif any(w in query_lower_en for w in ["appeal", "complain", "re-marking", "remarking", "petition", "grade dispute"]) or \
             any(w in query_orig_lower for w in ["ይግባኝ", "ቅሬታ", "ክስ", "ውጤት", "አቤቱታ", "ማመልከቻ", "እርምጃ"]):

            summary = "⚖️ **በዚህ ዩኒቨርሲቲ መመሪያ መሠረት፡ አንድ ተማሪ በተሰጠው ውጤት ወይም አስተዳደራዊ ውሳኔ ላይ ካልተስማማ የሪ-ማርኪንግ (Re-marking) ቅሬታ ወይም የይግባኝ አቤቱታ ማቅረብ መብቱ የተጠበቀ ነው!**"
            steps = [
                ("📝", "የይግባኝ ማመልከቻ ማዘጋጀት", "ተማሪው በውሳኔው ላይ ያለውን ቅሬታ በግልጽ የሚያስረዳ የተደገፈ የጽሑፍ ማመልከቻ (Petition) ያዘጋጃል።"),
                ("⏳", "በጊዜ ገደብ ማቅረብ", "ማመልከቻው ውጤቱ ከተለጠፈበት ቀን ጀምሮ በሕጉ በተደነገገው ቀናት ውስጥ ለሚመለከተው አካል መቅረብ አለበት።")
            ]
            response_text = generate_clean_student_response(university_choice, page_number, "ምዕራፍ 13 (የተማሪዎች የዲሲፕሊን እና የይግባኝ መመሪያ)", summary, steps, retrieved_text, user_lang)
            return response_text, final_lexical_pct, final_semantic_pct

        # 🛡️ Threshold Filter ማጣሪያ
        if final_semantic_pct < 65.0:
            if user_lang == 'am':
                msg = "🏛️ **የሕግ ምላሽ መድረክ**\n------------------\n⚠️ **ማሳሰቢያ፦**\n🔍 `ይህ መረጃ በዩኒቨርሲቲው መመሪያ ውስጥ አልተካተተም!`"
            else:
                msg = "🏛️ **Policy Response Portal**\n------------------\n⚠️ **Notice:**\n🔍 `This information is not included in the university policy.`"
            return msg, final_lexical_pct, final_semantic_pct

        # 5️⃣ ዲፎልት አቀራረብ
        else:
            summary = "የቀረበውን ጥያቄ መሠረት በማድረግ ሲስተሙ ተስማሚውን የሕግ አንቀጽ ከ PDF ላይ ፈልጎ አውጥቷል።"
            steps = [("🔍", "የተገኘው የሕግ መረጃ", "ለጥያቄዎ ተዛማጅ የሆነው የሕግ አንቀጽ ከዚህ በታች በንጹህ ትርጉም ተቀምጧል።")]
            response_text = generate_clean_student_response(university_choice, page_number, "የዩኒቨርሲቲ መመሪያ ሰነድ", summary, steps, retrieved_text, user_lang)
            return response_text, final_lexical_pct, final_semantic_pct

    except Exception as e:
        return f"❌ ስህተት አጋጥሟል፦ {e}", 0.0, 0.0


# ==========================================
# 🚀 4. እጅግ ማራኪ እና ዘመናዊ የ Gradio UI ፎርማት (በአዲሱ ርዕስ የተስተካከለ)
# ==========================================

with gr.Blocks(title="Smart Ethiopian University Policy Search Engine") as app:

    # ቪዥዋል ስታይሉ በማንኛውም ስሪት እንዲሰራ በ HTML <style> ማስተካከያ እዚህ ገብቷል
    gr.Markdown(
        """
        <style>
            body { background-color: #0f172a; }
            .gradio-container { max-width: 1100px !important; margin: auto; }
            .title-box { text-align: center; padding: 20px; color: #38bdf8; }
            .score-box { background-color: #1e293b; border-radius: 10px; padding: 15px; border: 1px solid #334155; }
        </style>

        <div class='title-box'>
            <h1>🏛️ Smart Ethiopian University Policy Search Engine</h1>
            <p style='color: #94a3b8; font-size: 16px;'>የኢትዮጵያ ዩኒቨርሲቲዎች የተማሪዎች አካዳሚክ እና ዲሲፕሊን መመሪያዎች ዘመናዊ የፍለጋ መድረክ</p>
        </div>
        """
    )

    with gr.Row():
        # የግራ ክፍያ - የግብዓት ሳጥኖች (Inputs)
        with gr.Column(scale=4):
            uni_input = gr.Dropdown(
                choices=university_list if 'university_list' in locals() else ["Gondar University"],
                label="🏢 ዩኒቨርሲቲ ይምረጡ (Select University)",
                value="Gondar University"
            )
            query_input = gr.Textbox(
                lines=4,
                placeholder="የሚፈልጉትን ሕግ እዚህ በንጹህ አማርኛ ወይም እንግሊዝኛ ይጠይቁ... (e.g., ፈተና ላይ ብታመም ምን ይደረጋል?)",
                label="💬 የተማሪዎች የጥያቄ ሳጥን (Student Query Box)"
            )

            # ሁለቱ አዝራሮች ጎን ለጎን እንዲቀመጡ በ Row ውስጥ አደረግናቸው
            with gr.Row():
                clear_btn = gr.Button("🗑️ አፅዳ (Clear)", variant="secondary")
                submit_btn = gr.Button("🔍 ሕጉን ፈልግ (Search Policy)", variant="primary")

            # ናሙና ጥያቄዎች (Examples) እንደ ሂንት ከታች ተቀምጠዋል
            gr.Examples(
                examples=[
                    ["Gondar University", "ተማሪዎች በካምፓስ ውስጥ ሰላማዊ ማህበር ለማደራጀት ወይም አቤቱታ (Petition) ለማቅረብ ያላቸው ሕጋዊ መብት ምንድነው?"],
                    ["Gondar University", "አንድ ተማሪ ፈተና ላይ ኮፒ ቢያደርግ ወይም የሌላ ሰውን ፕሮጀክት ቢሰርቅ የሚወሰድበት የዲሲፕሊን እርምጃ ምንድነው?"],
                    ["Gondar University", "በበሽታ ወይም በድንገተኛ አደጋ ምክንያት ዋና ፈተና ካመለጠኝ ማሟያ ፈተና (Makeup Exam) መውሰድ እችላለሁ?"],
                    ["Gondar University", "በአንድ ኮርስ ውጤት ላይ ቅሬታ ካለኝ የሪ-ማርኪንግ (Re-marking) ማመልከቻ እንዴት ማቅረብ እችላለሁ?"]
                ],
                inputs=[uni_input, query_input],
                label="💡 የተለመዱ የናሙና ጥያቄዎች ማሳያ (Sample Query Hints)"
            )

        # የቀኝ ክፍያ - የውጤት እና የነጥብ ማሳያ (Outputs & Scores)
        with gr.Column(scale=5):
            with gr.Group(elem_id="score-box-container"):
                gr.Markdown("### 📊 የፍለጋ ሞተር ትንተና ደረጃ (Search Confidence Analytics)")
                with gr.Row():
                    lexical_score = gr.Slider(minimum=0, maximum=100, label="🔍 ባህላዊ የቃላት ፍለጋ (Lexical Match Score)", interactive=False)
                    semantic_score = gr.Slider(minimum=0, maximum=100, label="🧠 የ AI ትርጉም ፍለጋ (Semantic AI Score)", interactive=False)

            gr.Markdown("<br>")
            # ዋናው የምላሽ ማሳያ ሳጥን
            output_markdown = gr.Markdown(label="🏛️ የሲስተም የሕግ ምላሽ ገፅ", value="*የፍለጋ ውጤቱ እዚህ ይወጣል...*")

    # 🔍 1. የሰርች አዝራር ሥራ ማስጀመር
    submit_btn.click(
        fn=campus_ir_pdf_rag_engine,
        inputs=[uni_input, query_input],
        outputs=[output_markdown, lexical_score, semantic_score]
    )

    # 🗑️ 2. የክሊር አዝራር ሥራ ማስጀመር (ሁሉንም በአንዴ ባዶ ያደርጋል)
    clear_btn.click(
        fn=lambda: ("", 0.0, 0.0, "*የፍለጋ ውጤቱ እዚህ ይወጣል...*"),
        inputs=None,
        outputs=[query_input, lexical_score, semantic_score, output_markdown]
    )

# ማስጀመር
app.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2e8270ae9897863fac.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
